# 1D CNN Approach

## Imports

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, confusion_matrix, precision_recall_curve, auc
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Dropout, Input

## Configuring Hyperparameters

In [ ]:
FS = 32
WINDOW_SEC = 60
STEP_SEC = 30

WINDOW_SIZE = FS * WINDOW_SEC
STEP_SIZE = FS * STEP_SEC

FEATURES = ['acc_mag', 'EDA', 'HR', 'TEMP']
TARGET = 'label'

## Loading Data and z-Score Normalization

In [ ]:
def load_and_normalize_data(folder_path):
    """
    Loads all nurse CSVs, applies subject-wise Z-score normalization,
    and stores them in a dictionary.
    """
    exclude_nurses = ['CE', 'EG']
    csv_files = glob.glob(os.path.join(folder_path, "processed_nurse_*.csv"))
    csv_files = [f for f in csv_files if not any(f"processed_nurse_{ex}.csv" in os.path.basename(f) for ex in exclude_nurses)]
    subjects_data = {}

    for file in csv_files:
        subject_id = os.path.basename(file).split('.')[0]
        df = pd.read_csv(file)

        # Binarize label: Assuming 0.0 is unstressed, >0 is stressed
        df[TARGET] = (df[TARGET] > 0).astype(int)

        # Subject-wise Z-Score Normalization
        scaler = StandardScaler()
        df[FEATURES] = scaler.fit_transform(df[FEATURES])

        subjects_data[subject_id] = df

    return subjects_data

## Sliding Window Epochs

In [ ]:
def create_windows(df):
    """
    Converts continuous normalized time-series into 3D tensors (Windows, Timesteps, Features).
    """
    data = df[FEATURES].values
    labels = df[TARGET].values

    X, y = [], []
    for start in range(0, len(data) - WINDOW_SIZE, STEP_SIZE):
        end = start + WINDOW_SIZE
        X.append(data[start:end])

        # Label for the window (e.g., mode/majority of the window's labels)
        window_labels = labels[start:end]
        majority_label = 1 if np.mean(window_labels) >= 0.5 else 0
        y.append(majority_label)

    return np.array(X), np.array(y)

## Moderate Undersampling

In [ ]:
def moderate_undersample(X, y, target_ratio=0.8):
    """
    Undersamples the majority class (Stressed=1) to reach a specific ratio (e.g., 80:20).
    """
    idx_stressed = np.where(y == 1)[0]
    idx_unstressed = np.where(y == 0)[0]

    # Calculate how many majority samples we need to hit the target_ratio
    # Equation: target_ratio = count_stressed / (count_stressed + count_unstressed)
    num_unstressed = len(idx_unstressed)

    if num_unstressed == 0:
        return X, y # Edge case fallback

    target_stressed_count = int((target_ratio / (1 - target_ratio)) * num_unstressed)

    # Don't oversample if we somehow have fewer stressed than the target requires
    target_stressed_count = min(target_stressed_count, len(idx_stressed))

    # Randomly select majority class indices
    sampled_stressed_idx = np.random.choice(idx_stressed, target_stressed_count, replace=False)

    # Combine and shuffle
    balanced_idx = np.concatenate([sampled_stressed_idx, idx_unstressed])
    np.random.shuffle(balanced_idx)

    return X[balanced_idx], y[balanced_idx]

## Model Creation

In [1]:
def build_1d_cnn(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(filters=32, kernel_size=5, activation='relu'),
        MaxPooling1D(pool_size=2),
        Conv1D(filters=64, kernel_size=3, activation='relu'),
        GlobalAveragePooling1D(),
        Dense(64, activation='relu', name='dense_1'),
        Dropout(0.4),
        Dense(1, activation='sigmoid', name='output_layer')
    ])

    # Focal Loss Setup (gamma=2.0 focuses on hard examples, alpha handles initial imbalance)
    focal_loss = tf.keras.losses.BinaryFocalCrossentropy(gamma=2.0, alpha=0.25)

    model.compile(optimizer='adam', loss=focal_loss, metrics=['accuracy'])
    return model

## Cross-Validation (LOSO) and Transfer Learning w/ Metrics

In [ ]:
def run_loso_pipeline(subjects_data):
    subject_ids = list(subjects_data.keys())

    for test_subject in subject_ids:
        print(f"\n{'='*40}\nLeaving out {test_subject} for Testing/Transfer Learning\n{'='*40}")

        # 1. Prepare Global Training Data (N-1 subjects)
        X_train_global, y_train_global = [], []
        for subj in subject_ids:
            if subj != test_subject:
                X_subj, y_subj = create_windows(subjects_data[subj])
                X_train_global.append(X_subj)
                y_train_global.append(y_subj)

        X_train_global = np.concatenate(X_train_global)
        y_train_global = np.concatenate(y_train_global)

        # Apply moderate undersampling to GLOBAL training set ONLY
        X_train_bal, y_train_bal = moderate_undersample(X_train_global, y_train_global, target_ratio=0.8)

        # 2. Train Global Model
        model = build_1d_cnn(input_shape=(WINDOW_SIZE, len(FEATURES)))
        print("Training Global Model...")
        model.fit(X_train_bal, y_train_bal, epochs=10, batch_size=64, verbose=0)

        # 3. Transfer Learning Prep (Chronological Split of Left-Out Subject)
        X_test_subj, y_test_subj = create_windows(subjects_data[test_subject])

        # Use first 20% of their shift for fine-tuning, test on remaining 80%
        split_idx = int(len(X_test_subj) * 0.20)
        X_finetune, y_finetune = X_test_subj[:split_idx], y_test_subj[:split_idx]
        X_eval, y_eval = X_test_subj[split_idx:], y_test_subj[split_idx:]

        # Undersample the fine-tuning set as well so it doesn't skew the transfer learning
        X_finetune_bal, y_finetune_bal = moderate_undersample(X_finetune, y_finetune, target_ratio=0.8)

        # Freeze Convolutional Layers
        for layer in model.layers:
            if isinstance(layer, Conv1D):
                layer.trainable = False

        # Recompile to apply frozen state
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                      loss=tf.keras.losses.BinaryFocalCrossentropy(gamma=2.0, alpha=0.25))

        # 4. Fine-Tune on Target Subject
        print("Fine-tuning on target subject's early data...")
        if len(X_finetune_bal) > 0:
            model.fit(X_finetune_bal, y_finetune_bal, epochs=5, batch_size=32, verbose=0)

        # 5. Evaluate on Target Subject's Remaining Data (The true test)
        y_pred_prob = model.predict(X_eval).flatten()
        y_pred_class = (y_pred_prob >= 0.5).astype(int)

        # 6. Calculate Specialized Metrics
        cm = confusion_matrix(y_eval, y_pred_class, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0,0,0,0)

        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        macro_f1 = f1_score(y_eval, y_pred_class, average='macro')

        # Minority F1 (Assuming 0 is the minority Unstressed class)
        minority_f1 = f1_score(y_eval, y_pred_class, pos_label=0, zero_division=0)

        # PR-AUC for the Minority Class
        # Note: precision_recall_curve expects probabilities for the positive class.
        # To get PR-AUC for class 0, we invert the true labels and probabilities.
        precision, recall, _ = precision_recall_curve((y_eval == 0).astype(int), 1 - y_pred_prob)
        pr_auc = auc(recall, precision)

        print("\n--- Evaluation Metrics ---")
        print("Confusion Matrix (TN, FP / FN, TP):\n", cm)
        print(f"Specificity (Unstressed Accuracy): {specificity:.4f}")
        print(f"Minority F1-Score (Unstressed):    {minority_f1:.4f}")
        print(f"Macro F1-Score:                    {macro_f1:.4f}")
        print(f"Minority PR-AUC:                   {pr_auc:.4f}")

## Run Model

In [ ]:
subjects_data = load_and_normalize_data("/Users/ericguan/Documents/GitHub/MLMAMidtermProject/data/Aditya")
run_loso_pipeline(subjects_data)